In [ ]:
import random
from datasets import load_dataset, Dataset
import os
import numpy as np

# 🌟 데이터셋 정보: youjunhyeok/Magpie-Qwen2-Pro-300K-Filtered-ko
# 📜 데이터셋 의미: 이 데이터셋은 한국어로 학습된 대규모 언어 모델(LLM)의 질의응답(QA) 및 텍스트 생성 능력을 향상시키기 위해 정제된 방대한 코퍼스입니다.
# 💡 활용 목표: 우리는 이 데이터를 활용해 모델이 어떤 유형의 질문(instruction)에 어떻게 답변했는지(response) 패턴을 파악하는 '프롬프트 탐색기'를 만들어 볼 거예요!

DATASET_NAME = "youjunhyeok/Magpie-Qwen2-Pro-300K-Filtered-ko"
SPLIT_NAME = 'train'
SAMPLE_COUNT = 10 # 🔥 초보자 실습을 위해 상위 10개 샘플만 살펴봅시다!

def load_dataset_safely(dataset_id: str, split: str, sample_count: int):
    """
    스트리밍 모드와 일반 모드를 모두 테스트하여 데이터를 로드하는 안전한 함수입니다.
    """
    print("========================================================")
    print(f"🚀 데이터셋 로딩을 시도합니다: {dataset_id}, Split: {split}")
    print("========================================================")
    
    # 1. 스트리밍 모드(streaming=True)로 로드를 시도합니다. (빠른 테스트용)
    try:
        dataset = load_dataset(dataset_id, split=split, streaming=True)
        print("✅ 성공! 스트리밍(Streaming) 모드로 데이터셋을 로드했습니다. (최대 효율!)")
    except Exception as e:
        print(f"⚠️ 스트리밍 로드 실패 ({e.__class__.__name__}). 일반 모드로 전환합니다.")
        # 스트리밍 실패 시, 일반 모드로 전환
        dataset = load_dataset(dataset_id, split=split, streaming=False)
        print("✅ 성공! 일반 Dataset 모드로 데이터셋을 로드했습니다.")
    
    # 2. 스트리밍 패턴을 사용하여 샘플링된 이터레이터를 준비합니다.
    if hasattr(dataset, "take"):
        # .take()가 존재하면 스트리밍 데이터셋(IterableDataset)로 간주
        print("\n✨ 패턴 인식: Streaming Dataset으로 추정되어 .take() 패턴을 사용합니다.")
        # next()를 사용하기 위해 list()로 한번 전개하는 것이 안전합니다.
        sampled_dataset_iterator = list(dataset.take(sample_count))
    else:
        # 일반 데이터셋 (Dataset)으로 간주
        print("\n✨ 패턴 인식: 일반 Dataset으로 추정되어 .take() 패턴을 사용합니다.")
        sampled_dataset_iterator = list(dataset.take(sample_count))
        
    return sampled_dataset_iterator

def analyze_dataset(sample_iterator):
    """
    샘플링된 데이터를 순회하며 창의적인 AI 분석을 수행하는 메인 함수입니다.
    """
    print("\n" + "="*80)
    print("💡 튜터의 관찰력 발휘! 🔥 샘플 분석을 시작합니다!")
    print("="*80)
    
    sample_list = sample_iterator
    if not sample_list:
        print("😭 분석할 샘플이 준비되지 않았습니다. 데이터를 확인해주세요.")
        return

    print(f"\n[✨ {len(sample_list)}개의 샘플을 분석하며 AI 데이터의 비밀을 파헤쳐 봅시다!]")
    print("-" * 50)

    analysis_results = []
    for i, sample in enumerate(sample_list):
        print(f"\n--- [분석 샘플 {i+1}/{len(sample_list)}]: 지식의 조각 발견! ---")
        
        # 1. 핵심 데이터 추출 및 해석 (Instruction & Response)
        instruction = sample.get('instruction', 'N/A')
        response = sample.get('response', 'N/A')
        
        # 2. 메타데이터 추출 (난이도, 의도 등)
        difficulty = sample.get('difficulty', 'Unknown')
        intent = sample.get('intent', 'General')
        task_category = sample.get('task_category', 'General')
        
        # 3. 창의적 활동: '프롬프트 완성도 점수' 매기기
        # 만약 난이도와 의도를 조합하여 이 샘플이 얼마나 잘 구조화되었는지 평가해봅니다.
        score = 0
        if difficulty in ['Easy', 'Medium', 'Hard']:
            score += 3
        if intent != 'General':
            score += 2
        if len(instruction) > 10 and len(response) > 10:
            score += 1

        analysis_results.append({
            "idx": i + 1,
            "instruction": instruction[:50] + "...",
            "response": response[:50] + "...",
            "difficulty": difficulty,
            "intent": intent,
            "score": score
        })

        print(f"📚 Instruction (질문 유형): {instruction[:30]}... (난이도: {difficulty})")
        print(f"🤖 Response (답변 내용): {response[:30]}... (의도: {intent})")
        
        # 🌟 [AI 튜터 시뮬레이션]: 이 샘플을 기반으로 다음 질문을 만들어봅시다!
        if difficulty == 'Hard' and intent == 'Knowledge':
            print(f"\n💡 튜터의 코멘트: 이 샘플은 '난이도: {difficulty}, 의도: {intent}'가 높아 매우 귀한 학습 데이터입니다!")
            print("🚀 [창의적 실습]: '이 주제에 대해 더 심화된 관점을 제시해주시겠어요?' 와 같은 후속 질문을 만들면 모델 능력을 테스트하기 좋습니다.")
        else:
            print("\n👍 [분석 결과]: 기본적인 QA 쌍이므로, 패턴 분석에 활용하기 좋습니다. 구조화가 잘 되어있네요!")


if __name__ == "__main__":
    # 1. 데이터 로드 (스트리밍/일반 모드 테스트 및 샘플링)
    sample_iterator = load_dataset_safely(DATASET_NAME, SPLIT_NAME, SAMPLE_COUNT)
    
    # 2. 분석 실행
    analyze_dataset(sample_iterator)

    print("\n========================================================")
    print("✨ 실습 완료! 정말 수고 많으셨어요, 코더님!")
    print("이처럼 데이터를 읽고, 메타 정보를 조합하며, 새로운 사용 시나리오를 만드는 과정이 바로 AI 엔지니어링의 핵심이랍니다!")
    print("다음에는 이 점수를 바탕으로 데이터셋의 빈도 분석을 해보면 더 재미있을 거예요! 😉")
    print("========================================================")